# Elasticsearch RAG Experiment

This notebook uses Elasticsearch as the retrieval layer for the course FAQ RAG assistant.

## 1. Setup

Keep the Elasticsearch Docker container running before executing this notebook.

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv
from openai import OpenAI
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not loaded from .env")

openai_client = OpenAI()

## 2. Check Elasticsearch

In [ ]:
response = requests.get("http://localhost:9200", timeout=10)
response.raise_for_status()
response.json()

## 3. Load FAQ Documents

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
len(documents), documents[0]

## 4. Create and Populate Elasticsearch Index

In [ ]:
from elastic_search import ElasticSearchIndex

es_index = ElasticSearchIndex(index_name="course-faq")
es_index.create_index(recreate=True)
es_index.index_documents(documents)

## 5. Test Retrieval

In [ ]:
question = "I just discovered the course. Can I join now?"

results = es_index.search(
    query=question,
    filter_dict={"course": "llm-zoomcamp"},
    boost_dict={"question": 3.0, "section": 0.5},
    num_results=5,
)

results[0]

## 6. Run RAG with Elasticsearch

In [ ]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=es_index,
    llm_client=openai_client,
)

answer = assistant.rag(question)
print(answer)

## 7. Compare with Minsearch

In [ ]:
from ingest import build_index

minsearch_index = build_index(documents)

minsearch_results = minsearch_index.search(
    query=question,
    filter_dict={"course": "llm-zoomcamp"},
    boost_dict={"question": 3.0, "section": 0.5},
    num_results=5,
)

minsearch_results[0]

In [ ]:
comparison = {
    "elasticsearch_top_question": results[0]["question"],
    "minsearch_top_question": minsearch_results[0]["question"],
}

comparison